In [ ]:
# @title 1 · ENVIRONMENT
# WEELLM STUDIO · COLAB NOTEBOOK · 2 CODE CELLS · HIDDEN BY DEFAULT (SHOW CODE TO INSPECT)
# RUN: RUNTIME → GPU → RUN ALL · CELL 1 INSTALLS · CELL 2 SERVES + HEARTBEATS
# GATED REPOS NEED HF TOKEN + AGREE ON THE REPO PAGE
# @title 1 · Environment setup
# ═══════════════════════════════════════════════════════════════
#  WeeLLM Studio · Cell 1 of 2 — Environment setup
#  Everything installs quietly; the ONLY output is the status box.
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, time, shutil, traceback, io, warnings, contextlib, logging
warnings.simplefilter('ignore')
# torchao ships Hopper-only / newer-CUDA .so files that can never load on a T4
# (it logs WARNING + falls back). Silence them: this cell must show ONLY the status box.
logging.getLogger('torchao').setLevel(logging.ERROR)

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

BOX_CSS = ('<style>'
 '.luxbox{max-width:700px;margin:18px auto;padding:26px 32px 22px;border-radius:14px;'
 'background:linear-gradient(170deg,#101014 0%,#16161D 55%,#0B0B10 100%);'
 'border:1px solid rgba(191,160,90,.38);color:#EDE6D3;'
 'font-family:"Helvetica Neue",-apple-system,"Segoe UI",Roboto,Arial,sans-serif;'
 'box-shadow:0 16px 50px rgba(0,0,0,.55);}'
 '.luxrule{height:3px;margin:0 0 16px;border-top:1px solid rgba(191,160,90,.65);border-bottom:1px solid rgba(191,160,90,.22);}'
 '.luxbox h2{margin:0 0 4px;font-family:Georgia,"Times New Roman",serif;font-weight:400;letter-spacing:.3em;font-size:14px;color:#C9A961;text-transform:uppercase;}'
 '.luxbox .sub{font-size:11px;color:#97907E;margin:0 0 14px;line-height:1.7;letter-spacing:.1em;text-transform:uppercase;}'
 '.luxrow{display:flex;justify-content:space-between;gap:14px;padding:8px 2px;border-top:1px solid rgba(235,230,211,.09);font-size:12.5px;}'
 '.luxrow span{color:#97907E;text-transform:uppercase;letter-spacing:.08em;font-size:11.5px;}'
 '.luxrow b{color:#F2ECDC;font-weight:600;text-align:right;text-transform:uppercase;letter-spacing:.06em;}'
 '.pill{display:inline-block;padding:3px 14px;border-radius:999px;font-size:10px;letter-spacing:.22em;font-weight:700;text-transform:uppercase;}'
 '.ok{background:transparent;color:#8FD6A4;border:1px solid rgba(110,180,130,.55);}'
 '.bad{background:transparent;color:#E89A9A;border:1px solid rgba(200,110,110,.6);}'
 '.run{background:transparent;color:#D9BE7A;border:1px solid rgba(191,160,90,.6);animation:luxpulse 1.6s ease-in-out infinite;}'
 '@keyframes luxpulse{0%,100%{opacity:1}50%{opacity:.4}}'
 '.luxnote{font-size:11px;color:#8A8474;margin-top:12px;line-height:1.7;letter-spacing:.06em;text-transform:uppercase;}'
 '</style>')

def _pill(kind, text):
    return "<span class='pill %s'>%s</span>" % (kind, text)

def _box_html(title, pill, sub, rows, note=''):
    body = ''.join("<div class='luxrow'><span>%s</span><b>%s</b></div>" % (k, v) for k, v in rows)
    if note:
        body += "<div class='luxnote'>%s</div>" % note
    return BOX_CSS + "<div class='luxbox'><div class='luxrule'></div><h2>%s &nbsp;%s</h2><p class='sub'>%s</p>%s</div>" % (title, pill, sub, body)

from IPython.display import display, HTML
_handle = display(HTML(_box_html('WeeLLM Studio · Environment', _pill('run', 'INSTALLING'), 'Cell 1 of 2 — installing packages, this takes a few minutes…', [])), display_id=True)
def _refresh(pill, sub, rows, note=''):
    _handle.update(HTML(_box_html('WeeLLM Studio · Environment', pill, sub, rows, note)))

results, ok_all, t0 = [], True, time.time()
def _torch_stack_cmd():
    # torch + torchvision + torchao must upgrade TOGETHER: pip pairs each
    # torchao with its torch, so upgrading torchao alone can never reach a
    # build that has FqnToConfig on old runtimes (and its .so files mismatch).
    # Everything is probed in FRESH subprocesses -- importing torch/torchao
    # here would cache the OLD build in this kernel, and verification would
    # keep seeing the old one even after pip replaces the files on disk.
    probe = subprocess.run(
        [sys.executable, '-c',
         'try:\n import torchao\n from torchao import quantization as _q\n'
         ' print("OK" if hasattr(_q, "FqnToConfig") else "OLD")\n'
         'except Exception:\n print("MISSING")'],
        capture_output=True, text=True, timeout=300)
    if (probe.stdout or '').strip().endswith('OK'):
        return [sys.executable, '-c', 'print("torch stack already OK, skip")']
    cuda = subprocess.run(
        [sys.executable, '-c',
         'try:\n import torch\n print(torch.version.cuda or "cpu")\n'
         'except Exception:\n print("cpu")'],
        capture_output=True, text=True, timeout=300)
    tag = (cuda.stdout or '').strip().split('.')
    tag = ('cu' + tag[0] + tag[1]) if len(tag) >= 2 and tag[0].isdigit() else 'cpu'
    idx = 'https://download.pytorch.org/whl/' + tag
    return [sys.executable, '-c',
            'import subprocess, sys;'
            ' pkgs = ["torch", "torchvision", "torchao"];'
            ' idx = %r;'
            ' r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs + ["--index-url", idx]);'
            ' raise SystemExit(0 if r.returncode == 0 else'
            ' subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs).returncode)'
            % idx]
STEPS = [
    ('Base tools (hf_transfer · gradio · peft)', [sys.executable, '-m', 'pip', 'install', '-q', 'hf_transfer', 'psutil', 'gradio>=5', 'peft>=0.10.0']),
    ('torch stack (torch+torchvision+torchao, matched CUDA)', _torch_stack_cmd()),
    ('diffusers @ git (Z-Image · Krea · LTX support)', [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/huggingface/diffusers.git']),
    ('WeeLLM-Enhanced @ git (forced fresh, no dep churn)', [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps', 'git+https://github.com/GodL-x-SouL/WeeLLM-Enhanced.git']),
]
for name, cmd in STEPS:
    _refresh(_pill('run', 'INSTALLING'), 'Running: %s …' % name, results)
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=1500)
        ok = (p.returncode == 0)
        detail = 'installed' if ok else ('exit %d · %s' % (p.returncode, (p.stderr or '')[-300:]))
    except Exception as e:
        ok, detail = False, '%s: %s' % (type(e).__name__, str(e)[-200:])
    ok_all = ok_all and ok
    results.append((name, 'done' if ok else ('FAILED — ' + detail)))
    _refresh(_pill('run', 'INSTALLING'), 'Running setup…', results)

_refresh(_pill('run', 'INSTALLING'), 'Verifying packages (imports run silenced)…', results)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        try:
            import torch, gradio, diffusers, huggingface_hub
            from importlib.metadata import version as _v
            try:
                import torchao as _ta
                _ta_v = getattr(_ta, '__version__', '?')
                from torchao import quantization as _tq
                assert hasattr(_tq, 'FqnToConfig'), 'torchao %s has no quantization.FqnToConfig (need torchao>=0.15)' % _ta_v
                results.append(('torchao ' + str(_ta_v) + ' · FqnToConfig', 'ok'))
            except Exception as _te:
                ok_all = False
                results.append(('torchao check', 'FAILED — %s: %s (fix: pip install -U torchao)' % (type(_te).__name__, str(_te)[-220:])))
                raise
            try:
                import weellm
            except Exception as _we:
                ok_all = False
                results.append(('weellm import', 'FAILED — %s: %s' % (type(_we).__name__, str(_we)[-220:])))
                raise
            gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime → GPU'
            if not torch.cuda.is_available():
                ok_all = False
            free_gb = shutil.disk_usage('/content').free / 1e9
            results += [('torch ' + torch.__version__, gpu), ('gradio ' + _v('gradio') + ' · diffusers ' + diffusers.__version__ + ' · weellm ' + weellm.__version__, 'ok'), ('disk free %.0f GB · setup %.0fs' % (free_gb, time.time() - t0), 'ok')]
            assert hasattr(diffusers, 'ZImagePipeline'), 'diffusers has no ZImagePipeline'
        except Exception:
            ok_all = False
            results.append(('Verification', 'FAILED — ' + traceback.format_exc(limit=5)[-600:]))

os.makedirs('/content/weellm_studio', exist_ok=True)
if ok_all:
    _refresh(_pill('ok', 'READY'), 'Cell 1 of 2 done — run Cell 2 to open the app.', results)
else:
    _refresh(_pill('bad', 'FAILED'), 'Something failed — read the rows above, fix, and re-run this cell.', results)


In [ ]:
# @title 2 · STUDIO
# WEELLM STUDIO · APP CELL · KEEP RUNNING — HEARTBEAT HOLDS THE SESSION
# @title 2 · WeeLLM Studio
# ═══════════════════════════════════════════════════════════════
#  WeeLLM Studio · Cell 2 of 2 — Cinematic Gradio app
#  Only deliberate UI boxes are ever displayed. All launcher noise
#  (gradio URLs, tunnel chatter, warnings) is captured, so nothing
#  leaks outside the boxes except the final share-link card.
# ═══════════════════════════════════════════════════════════════
import os, sys, io, gc, time, math, shutil, traceback, threading, contextlib, warnings, logging, inspect, fnmatch, random
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
warnings.filterwarnings('ignore')
import torch
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        import gradio as gr  # pre-import silenced: later `import gradio` statements become no-ops
from IPython.display import display, HTML

if not torch.cuda.is_available():
    display(HTML("<div style='max-width:640px;margin:16px auto;padding:20px;border-radius:14px;background:#160d0d;border:1px solid #a33;color:#ffd7d7;font-family:sans-serif'>No GPU detected. Set <b>Runtime → Change runtime type → GPU</b> and re-run.</div>"))
    raise SystemExit('WeeLLM Studio needs a GPU runtime.')

# ── luxury shell (notebook boxes) ──────────────────────────────
SHELL_CSS = ('<style>.luxbox{max-width:700px;margin:16px auto;padding:24px 30px 20px;border-radius:14px;'
 'background:linear-gradient(170deg,#101014 0%,#16161D 55%,#0B0B10 100%);'
 'border:1px solid rgba(191,160,90,.38);color:#EDE6D3;'
 'font-family:"Helvetica Neue",-apple-system,"Segoe UI",Roboto,Arial,sans-serif;'
 'box-shadow:0 16px 50px rgba(0,0,0,.55);}'
 '.luxrule{height:3px;margin:0 0 14px;border-top:1px solid rgba(191,160,90,.65);border-bottom:1px solid rgba(191,160,90,.22);}'
 '.luxbox h2{margin:0 0 3px;font-family:Georgia,"Times New Roman",serif;font-weight:400;letter-spacing:.3em;font-size:15px;color:#C9A961;text-transform:uppercase;}'
 '.luxbox .sub{font-size:11px;color:#97907E;margin:0 0 6px;line-height:1.7;letter-spacing:.1em;text-transform:uppercase;}'
 '.pill{display:inline-block;padding:3px 14px;border-radius:999px;font-size:10px;letter-spacing:.22em;font-weight:700;text-transform:uppercase;}'
 '.ok{background:transparent;color:#8FD6A4;border:1px solid rgba(110,180,130,.55);}'
 '.bad{background:transparent;color:#E89A9A;border:1px solid rgba(200,110,110,.6);}'
 '.run{background:transparent;color:#D9BE7A;border:1px solid rgba(191,160,90,.6);animation:luxpulse 1.6s ease-in-out infinite;}'
 '@keyframes luxpulse{0%,100%{opacity:1}50%{opacity:.4}}'
 'a.goldbtn{display:inline-block;margin-top:14px;padding:12px 36px;border-radius:999px;text-decoration:none;font-weight:700;'
 'letter-spacing:.24em;font-size:12px;color:#0E0D0A;background:linear-gradient(135deg,#D9BE7A,#A88436);'
 'border:1px solid rgba(240,220,160,.5);text-transform:uppercase;}'
 '.mono{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;color:#B9B2A0;word-break:break-all;}'
 '</style>')

def shell_box(title, pill, sub, link=None, handle=None):
    btn = ("<br><a class='goldbtn' href='%s' target='_blank'>OPEN WEELLM STUDIO</a><p class='mono'>%s</p>" % (link, link)) if link else ''
    html = HTML(SHELL_CSS + "<div class='luxbox'><div class='luxrule'></div><h2>%s &nbsp;%s</h2><p class='sub'>%s</p>%s</div>" % (title, pill, sub, btn))
    if handle is None:
        return display(html, display_id=True)
    handle.update(html)
    return handle

# ── model registry (every filename verified on HF Hub) ─────────
# quant value 'full' = whole-repo BF16 (or official low-bit release for Ideogram);
# otherwise value = GGUF filename inside gguf_repo (diffusion transformer only).
TASKS = [('TEXT → IMAGE', 't2i'), ('IMAGE EDIT', 'edit'), ('TEXT → VIDEO', 't2v'), ('LTX 2.5 · IMAGE → VIDEO', 'i2v')]
MODELS = {
 'flux2-klein-9b': dict(task='t2i', label='FLUX.2-Klein 9B', pipe='t2i',
  repo_url='https://huggingface.co/black-forest-labs/FLUX.2-klein-9B', mirror_url=None, gated=True,
  access='Gated repo — token required (even for quants: the ~16 GB encoder + VAE come from here). Click Agree on the repo page.',
  gguf_repo_url='https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF',
  steps=4, guid=0.0, w=1024, h=1024,
  ignore=['flux-2-klein-9b.safetensors', '*.jpg'],
  quants=[
   ('full', 'Full BF16 · official repo · gated · ~35 GB', 'https://huggingface.co/black-forest-labs/FLUX.2-klein-9B'),
   ('flux-2-klein-9b-BF16.gguf', 'Full BF16 GGUF · 18.2 GB · no login', 'https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/blob/main/flux-2-klein-9b-BF16.gguf'),
   ('flux-2-klein-9b-Q4_K_M.gguf', 'Q4_K_M · 5.9 GB · recommended', 'https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/blob/main/flux-2-klein-9b-Q4_K_M.gguf'),
   ('flux-2-klein-9b-Q6_K.gguf', 'Q6_K · 7.9 GB', 'https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/blob/main/flux-2-klein-9b-Q6_K.gguf'),
   ('flux-2-klein-9b-Q8_0.gguf', 'Q8_0 · 10.0 GB', 'https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/blob/main/flux-2-klein-9b-Q8_0.gguf'),
  ],
  default_q='flux-2-klein-9b-Q4_K_M.gguf'),
 'z-image-turbo': dict(task='t2i', label='Z-Image-Turbo', pipe='t2i',
  repo_url='https://huggingface.co/Tongyi-MAI/Z-Image-Turbo', mirror_url=None, gated=False,
  access='Open repo — no token needed.',
  gguf_repo_url='https://huggingface.co/gguf-org/z-image-gguf',
  steps=8, guid=1.0, w=1024, h=1024,
  ignore=[],
  quants=[
   ('full', 'Full BF16 · 32.9 GB · open', 'https://huggingface.co/Tongyi-MAI/Z-Image-Turbo'),
   ('z-image-turbo-q4_k_m.gguf', 'Q4_K_M · 4.5 GB · recommended', 'https://huggingface.co/gguf-org/z-image-gguf/blob/main/z-image-turbo-q4_k_m.gguf'),
   ('z-image-turbo-q6_k.gguf', 'Q6_K · 5.9 GB', 'https://huggingface.co/gguf-org/z-image-gguf/blob/main/z-image-turbo-q6_k.gguf'),
   ('z-image-turbo-q8_0.gguf', 'Q8_0 · 7.2 GB', 'https://huggingface.co/gguf-org/z-image-gguf/blob/main/z-image-turbo-q8_0.gguf'),
  ],
  default_q='z-image-turbo-q4_k_m.gguf'),
 'krea-2-turbo': dict(task='t2i', label='Krea 2 Turbo', pipe='t2i',
  repo_url='https://huggingface.co/krea/Krea-2-Turbo', mirror_url='https://huggingface.co/neuralnetworker/Krea-2-Turbo', gated=True,
  access='Official repo is gated, but the open mirror is auto-used when no token is given.',
  gguf_repo_url='https://huggingface.co/vantagewithai/Krea-2-Turbo-GGUF',
  steps=8, guid=0.0, w=1024, h=1024,
  ignore=['turbo.safetensors', 'images/*'],
  quants=[
   ('full', 'Full BF16 · gated (mirror auto-used w/o token)', 'https://huggingface.co/krea/Krea-2-Turbo'),
   ('krea2_turbo-Q4_K_M.gguf', 'Q4_K_M · 7.5 GB · recommended', 'https://huggingface.co/vantagewithai/Krea-2-Turbo-GGUF/blob/main/krea2_turbo-Q4_K_M.gguf'),
   ('krea2_turbo-Q6_K.gguf', 'Q6_K · 10.6 GB', 'https://huggingface.co/vantagewithai/Krea-2-Turbo-GGUF/blob/main/krea2_turbo-Q6_K.gguf'),
   ('krea2_turbo-Q8_0.gguf', 'Q8_0 · 13.7 GB', 'https://huggingface.co/vantagewithai/Krea-2-Turbo-GGUF/blob/main/krea2_turbo-Q8_0.gguf'),
  ],
  default_q='krea2_turbo-Q4_K_M.gguf'),
 'ideogram-4': dict(task='t2i', label='Ideogram 4', pipe='t2i',
  repo_url='https://huggingface.co/ideogram-ai/ideogram-4-fp8', mirror_url=None, gated=True,
  access='Gated repo — token required (even for quants: encoder + VAE come from here). One GGUF serves both transformer branches.',
  gguf_repo_url='https://huggingface.co/Abiray/ideogram-4-GGUF',
  steps=30, guid=0.0, w=1024, h=1024,
  ignore=['assets/*'],
  quants=[
   ('full', 'Official FP8 release · gated', 'https://huggingface.co/ideogram-ai/ideogram-4-fp8'),
   ('ideogram4-Q4_K.gguf', 'Q4_K · 5.8 GB · minimum (no _M exists)', 'https://huggingface.co/Abiray/ideogram-4-GGUF/blob/main/ideogram4-Q4_K.gguf'),
   ('ideogram4-Q6_K.gguf', 'Q6_K · 8.0 GB · recommended', 'https://huggingface.co/Abiray/ideogram-4-GGUF/blob/main/ideogram4-Q6_K.gguf'),
   ('ideogram4-Q8_0.gguf', 'Q8_0 · 10.1 GB', 'https://huggingface.co/Abiray/ideogram-4-GGUF/blob/main/ideogram4-Q8_0.gguf'),
  ],
  default_q='ideogram4-Q6_K.gguf'),
 'ernie-turbo': dict(task='t2i', label='ERNIE-Image-Turbo', pipe='t2i',
  repo_url='https://huggingface.co/baidu/ERNIE-Image-Turbo', mirror_url=None, gated=False,
  access='Open repo — no token needed.',
  gguf_repo_url='https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF',
  steps=5, guid=0.0, w=1024, h=1024,
  ignore=[],
  quants=[
   ('full', 'Full BF16 · ~31 GB · open', 'https://huggingface.co/baidu/ERNIE-Image-Turbo'),
   ('ernie-image-turbo-Q4_K_M.gguf', 'Q4_K_M · 5.0 GB · recommended', 'https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/blob/main/ernie-image-turbo-Q4_K_M.gguf'),
   ('ernie-image-turbo-Q6_K.gguf', 'Q6_K · 6.8 GB', 'https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/blob/main/ernie-image-turbo-Q6_K.gguf'),
   ('ernie-image-turbo-Q8_0.gguf', 'Q8_0 · 8.7 GB', 'https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/blob/main/ernie-image-turbo-Q8_0.gguf'),
  ],
  default_q='ernie-image-turbo-Q4_K_M.gguf'),
 'qwen-image-edit': dict(task='edit', label='Qwen-Image-Edit', pipe='edit',
  repo_url='https://huggingface.co/Qwen/Qwen-Image-Edit', mirror_url=None, gated=False,
  access='Open repo — no token needed.',
  gguf_repo_url='https://huggingface.co/QuantStack/Qwen-Image-Edit-GGUF',
  strength=0.8,
  steps=15, guid=0.0, w=1024, h=1024,
  ignore=[],
  quants=[
   ('full', 'Full BF16 · 57.7 GB · open', 'https://huggingface.co/Qwen/Qwen-Image-Edit'),
   ('Qwen_Image_Edit-Q4_K_M.gguf', 'Q4_K_M · 13.1 GB · recommended', 'https://huggingface.co/QuantStack/Qwen-Image-Edit-GGUF/blob/main/Qwen_Image_Edit-Q4_K_M.gguf'),
   ('Qwen_Image_Edit-Q6_K.gguf', 'Q6_K · 16.8 GB', 'https://huggingface.co/QuantStack/Qwen-Image-Edit-GGUF/blob/main/Qwen_Image_Edit-Q6_K.gguf'),
   ('Qwen_Image_Edit-Q8_0.gguf', 'Q8_0 · 21.8 GB', 'https://huggingface.co/QuantStack/Qwen-Image-Edit-GGUF/blob/main/Qwen_Image_Edit-Q8_0.gguf'),
  ],
  default_q='Qwen_Image_Edit-Q4_K_M.gguf'),
 'minimax-h3': dict(task='t2v', label='MiniMax-H3 (FL2VA) · experimental', pipe='t2v',
  repo_url='https://huggingface.co/MiniMaxAI/MiniMax-H3', mirror_url=None, gated=False,
  access='Open repo — experimental: pruned-GGUF path only; full weights (144 GB FL2VA) cannot fit Colab.',
  gguf_repo_url='https://huggingface.co/Abiray/MiniMax-H3-Pruned-GGUF',
  te_gguf=('https://huggingface.co/DeepBeepMeep/MiniMax-H3', 'Qwen3-VL-32B-Instruct/qwen3vl-32B-MiniMax-H3-Q4_K_M.gguf'), subfolder='FL2VA', fetch_patterns=['*.json', 'FL2VA/*.json', 'FL2VA/tokenizer*', 'FL2VA/tokenizer/*', 'FL2VA/processor/*', 'FL2VA/visual_vae/*', 'FL2VA/audio_vae/*', 'FL2VA/scheduler/*', 'FL2VA/*.py', 'FL2VA/*.txt'],
  steps=6, guid=0.0, w=960, h=544, frames=73,
  ignore=[],
  quants=[
   ('MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf', 'Q4_K_M pruned · 11.6 GB · recommended', 'https://huggingface.co/Abiray/MiniMax-H3-Pruned-GGUF/blob/main/MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf'),
   ('MiniMax-H3-FL2VA-Pruned-Q6_K.gguf', 'Q6_K pruned · 16.7 GB', 'https://huggingface.co/Abiray/MiniMax-H3-Pruned-GGUF/blob/main/MiniMax-H3-FL2VA-Pruned-Q6_K.gguf'),
   ('MiniMax-H3-FL2VA-Pruned-Q8_0.gguf', 'Q8_0 pruned · 21.6 GB', 'https://huggingface.co/Abiray/MiniMax-H3-Pruned-GGUF/blob/main/MiniMax-H3-FL2VA-Pruned-Q8_0.gguf'),
  ],
  default_q='MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf'),
 'ltx25': dict(task='i2v', label='LTX 2.5 · distilled', pipe='i2v',
  repo_url='https://huggingface.co/Lightricks/LTX-2.5-Diffusers', mirror_url=None, gated=True,
  access='Gated repo — token required (even for quants: the encoder stack + VAEs come from here). Click Agree and Access on the repo page.',
  gguf_repo_url='https://huggingface.co/joeygambino/LTX-2.5-Quantized',
  ignore=['ltx-2.5-*.safetensors', '*.webp', 'transformer_full/*', 'loras/*', 'latent_upscale_models/*', 'model_patches/*'],
  steps=25, guid=0.0, w=960, h=544, frames=121,
  quants=[('LTX25-distilled-DiT-Q4_K_M.gguf', 'Q4_K_M · 14.2 GB · recommended · ~55 GB total', 'https://huggingface.co/joeygambino/LTX-2.5-Quantized/blob/main/LTX25-distilled-DiT-Q4_K_M.gguf'),
   ('LTX25-distilled-DiT-Q6_K.gguf', 'Q6_K · 17.7 GB · ~58 GB total', 'https://huggingface.co/joeygambino/LTX-2.5-Quantized/blob/main/LTX25-distilled-DiT-Q6_K.gguf')],
  default_q='LTX25-distilled-DiT-Q4_K_M.gguf'),
 'ltx25-w4a8': dict(task='i2v', label='LTX 2.5 W4A8 ConvRot', pipe='i2v',
  repo_url='https://huggingface.co/Lightricks/LTX-2.5-Diffusers', mirror_url=None, gated=True,
  access='Gated base repo (token + Agree) supplies VAEs/configs. The W4A8 DiT + encoder come from the open Winnougan pack; WeeLLM decodes them block-by-block in pure torch (bit-identical to ComfyUI). T4-friendly.',
  gguf_repo_url='https://huggingface.co/Winnougan/ltx-2.5-w4a8-convrot-int4-convrot-Winnougan-Blessing',
  te_safetensors=('https://huggingface.co/Winnougan/ltx-2.5-w4a8-convrot-int4-convrot-Winnougan-Blessing', 'text_encoders/gemma4-12b-with-proj-ltx-2.5-w4a8_convrot.safetensors'),
  ignore=['ltx-2.5-*.safetensors', '*.webp', 'transformer_full/*', 'loras/*', 'latent_upscale_models/*', 'model_patches/*'],
  steps=25, guid=0.0, w=960, h=544, frames=121,
  quants=[('diffusion_models/ltx-2.5-22b-distilled-transformer-w4a8_convrot.safetensors', 'W4A8 ConvRot DiT · 12.5 GB · recommended (pairs with the W4A8 encoder below)', 'https://huggingface.co/Winnougan/ltx-2.5-w4a8-convrot-int4-convrot-Winnougan-Blessing/blob/main/diffusion_models/ltx-2.5-22b-distilled-transformer-w4a8_convrot.safetensors')],
  default_q='diffusion_models/ltx-2.5-22b-distilled-transformer-w4a8_convrot.safetensors'),
 'ltx25-int8': dict(task='i2v', label='LTX 2.5 INT8 ConvRot (official)', pipe='i2v',
  repo_url='https://huggingface.co/Lightricks/LTX-2.5-Diffusers', mirror_url=None, gated=True,
  access='Fully gated path — token + Agree needed for the base repo AND the official INT8 files. WeeLLM decodes INT8 (incl. ConvRot rotation) in pure torch; natively fast on T4 INT8 cores.',
  gguf_repo_url='https://huggingface.co/Lightricks/LTX-2.5',
  te_safetensors=('https://huggingface.co/Lightricks/LTX-2.5', 'text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors'),
  ignore=['ltx-2.5-*.safetensors', '*.webp', 'transformer_full/*', 'loras/*', 'latent_upscale_models/*', 'model_patches/*'],
  steps=25, guid=0.0, w=960, h=544, frames=121,
  quants=[('diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors', 'INT8 ConvRot DiT · official · recommended (pairs with the INT8 encoder below)', 'https://huggingface.co/Lightricks/LTX-2.5/blob/main/diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors')],
  default_q='diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'),

}
PIPE_CLS = {'t2i': 'WeeTextToImagePipeline', 'edit': 'WeeImageToImagePipeline', 't2v': 'WeeVideoPipeline', 'i2v': 'WeeVideoPipeline'}
def _rid(url):
    u = url.split('huggingface.co/', 1)[-1]
    for k in ('/blob/', '/tree/', '/resolve/'):
        u = u.split(k)[0]
    return u.strip('/')
def _rfile(url):
    for k in ('/blob/main/', '/resolve/main/'):
        if k in url:
            return url.split(k)[-1]
    return url.rsplit('/', 1)[-1]
OUT_DIR = '/content/weellm_studio'
os.makedirs(OUT_DIR, exist_ok=True)

# ── realtime log bus (feeds the in-app logs panel) ─────────────
LOG_BUF, LOG_LOCK = [], threading.Lock()
def _push_log(msg):
    with LOG_LOCK:
        LOG_BUF.append(msg)
        del LOG_BUF[:-400]
def _log_text():
    with LOG_LOCK:
        return '\n'.join(LOG_BUF)[-12000:] or '— LOGS STREAM HERE DURING DOWNLOAD AND GENERATION —'
class _BusHandler(logging.Handler):
    def emit(self, record):
        try:
            _push_log('%s: %s' % (record.name, record.getMessage()))
        except Exception:
            pass
for _lname in ('weellm', 'weellm.cache'):
    _lg = logging.getLogger(_lname)
    _lg.setLevel(logging.INFO)
    _lg.propagate = False
    if not any(isinstance(h, _BusHandler) for h in _lg.handlers):
        _lg.addHandler(_BusHandler())
def ustatus(msg):
    _push_log('[studio] ' + msg)

# ── helpers ────────────────────────────────────────────────────
def _fmt_gb(n):
    return '%.1f GB' % (n / 1e9) if n >= 1e9 else '%d MB' % (n / 1e6)
def _budgets():
    import psutil
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    ram = psutil.virtual_memory().total / 1e9
    return round(max(4.0, vram - 3.0), 1), round(max(4.0, min(10.0, ram - 4.0)), 1)
def _hub_dir(repo):
    from huggingface_hub import constants
    root = os.environ.get('HF_HUB_CACHE', constants.HF_HUB_CACHE)
    return os.path.join(root, 'models--' + repo.replace('/', '--'), 'blobs')
_TEMP_SUFFIXES = ('.part', '.incomplete', '.lock', '.tmp')
def _du(path):
    t = 0
    for dp, _dn, fn in os.walk(path):
        for f in fn:
            if f.startswith('tmp') or f.endswith(_TEMP_SUFFIXES):
                continue  # in-flight transfer fragments, not real bytes yet
            try:
                t += os.path.getsize(os.path.join(dp, f))
            except OSError:
                pass
    return t
def _repo_bytes(repo, token, patterns=None, exclude=()):
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        info = api.model_info(repo, token=token or None, files_metadata=True)
    except TypeError:
        info = api.model_info(repo, token=token or None)
    total, n = 0, 0
    for s in info.siblings:
        name = s.rfilename
        sz = getattr(s, 'size', None) or 0
        if patterns and not any(fnmatch.fnmatch(name, p) for p in patterns):
            continue
        if any(fnmatch.fnmatch(name, e) for e in exclude):
            continue
        total += sz
        n += 1
    return total, n


def _gguf_bytes(repo, filename, token):
    return _repo_bytes(repo, token, patterns=[filename])
def _snap73(n):
    n = max(9, int(n))
    k = max(0, round((n - 5) / 17))
    return 17 * k + 5
def _snap121(n):
    n = max(9, int(n))
    return max(1, ((n - 1 + 4) // 8) * 8 + 1)

# ── download plan + worker (HF Hub only, real byte progress) ───
def _cached_plan_bytes(repo, patterns, exclude, ggufs, token):
    """Bytes of this exact plan already sitting in the local HF cache.

    Reruns must not be told to need the full total again. Base repo: walk
    only the NEWEST snapshot dir (stale revisions do not count) and sum
    planned files, deduped by inode so shared blobs count once. Single
    quant files: ask the Hub cache without downloading anything.
    """
    cached, seen = 0, set()

    def _add(path):
        nonlocal cached
        try:
            st = os.stat(path)
        except OSError:
            return
        key = (st.st_dev, st.st_ino)
        if key in seen:
            return
        seen.add(key)
        cached += st.st_size

    try:
        from huggingface_hub import constants
        root = os.environ.get("HF_HUB_CACHE", constants.HF_HUB_CACHE)
        snap_root = os.path.join(root, "models--" + repo.replace("/", "--"), "snapshots")
        if os.path.isdir(snap_root):
            snaps = [os.path.join(snap_root, s) for s in os.listdir(snap_root)]
            snaps = [s for s in snaps if os.path.isdir(s)]
            if snaps:
                newest = max(snaps, key=lambda s: os.path.getmtime(s))
                for dp, _dn, fn in os.walk(newest):
                    for f in fn:
                        p = os.path.join(dp, f)
                        try:
                            rel = os.path.relpath(p, newest)
                        except ValueError:
                            continue
                        if patterns and not any(fnmatch.fnmatch(rel, pat) for pat in patterns):
                            continue
                        if exclude and any(fnmatch.fnmatch(rel, e) for e in exclude):
                            continue
                        _add(p)
    except Exception:
        pass
    try:
        from huggingface_hub import hf_hub_download
        for grepo, gfile in (ggufs or []):
            try:
                lp = hf_hub_download(repo_id=grepo, filename=gfile,
                                     token=token or None, local_files_only=True)
                _add(lp)
            except Exception:
                pass
    except Exception:
        pass
    return cached


def plan_download(entry, quant, token):
    repo = _rid(entry['repo_url'])
    if entry.get('mirror_url') and not (token or '').strip():
        repo = _rid(entry['mirror_url'])  # non-gated mirror auto-preferred without a token
    ignore = list(entry.get('ignore', []))
    parts, ggufs = [], []
    if quant == 'full':
        total, n = _repo_bytes(repo, token, patterns=entry.get('fetch_patterns'), exclude=ignore)
        parts.append(('%s full repo (%d files)' % (repo, n), total, 'https://huggingface.co/' + repo))
    else:
        ignore = ignore + ['transformer/*.safetensors', 'unconditional_transformer/*.safetensors']
        # COLAB SLIM (free-tier disk): skip weights that are never executed.
        # - prompt_enhancer: the LTX video adapter hard-skips it (dummy part).
        # - text_encoder weights: only when a quant encoder replaces them
        #   (te_gguf / te_safetensors). Configs + tokenizers (*.json) still fetch.
        if entry.get('pipe') == 'i2v':
            ignore = ignore + ['prompt_enhancer/*.safetensors']
        if entry.get('te_gguf') or entry.get('te_safetensors'):
            ignore = ignore + ['text_encoder/*.safetensors', 'text_encoder_2/*.safetensors',
                               'text_encoder_3/*.safetensors', 'text_encoder_4/*.safetensors']
        grepo = _rid(entry['gguf_repo_url'])
        gsize, _ = _gguf_bytes(grepo, quant, token)
        if gsize == 0:
            raise RuntimeError('Quant file not found on Hub: %s/%s' % (grepo, quant))
        ggufs.append((grepo, quant))
        base_total, _ = _repo_bytes(repo, token, patterns=entry.get('fetch_patterns'), exclude=['transformer/*.safetensors', 'unconditional_transformer/*.safetensors'] + ignore)
        parts.append(('Base %s : encoder + VAE + configs (transformer skipped)' % repo, base_total, 'https://huggingface.co/' + repo))
        parts.append(('Transformer ' + quant, gsize, entry['gguf_repo_url'] + '/blob/main/' + quant))
        if entry.get('te_gguf'):
            trepo = _rid(entry['te_gguf'][0])
            tsize, _ = _gguf_bytes(trepo, entry['te_gguf'][1], token)
            ggufs.append((trepo, entry['te_gguf'][1]))
            parts.append(('Text-encoder ' + entry['te_gguf'][1].rsplit('/', 1)[-1], tsize, entry['te_gguf'][0] + '/blob/main/' + entry['te_gguf'][1]))
        if entry.get('te_safetensors'):
            trepo = _rid(entry['te_safetensors'][0])
            tsize, _ = _gguf_bytes(trepo, entry['te_safetensors'][1], token)
            if tsize == 0:
                raise RuntimeError('Quant file not found on Hub: %s/%s' % (trepo, entry['te_safetensors'][1]))
            ggufs.append((trepo, entry['te_safetensors'][1]))
            parts.append(('Text-encoder ' + entry['te_safetensors'][1].rsplit('/', 1)[-1], tsize, entry['te_safetensors'][0] + '/blob/main/' + entry['te_safetensors'][1]))
        total = sum(p[1] for p in parts)
    _excl = list(ignore) if quant == 'full' else (['transformer/*.safetensors', 'unconditional_transformer/*.safetensors'] + ignore)
    cached = _cached_plan_bytes(repo, entry.get('fetch_patterns'), _excl, ggufs, token)
    need = max(0, total - cached)
    free = shutil.disk_usage(os.path.expanduser('~/.cache/huggingface')).free
    if need > free * 0.92:
        raise RuntimeError('Not enough disk: need %s more (%s total, %s already cached), have %s free. Pick a smaller quant or free space.' % (_fmt_gb(need), _fmt_gb(total), _fmt_gb(cached), _fmt_gb(free)))
    if cached > 0:
        ustatus('preflight: %s already cached, only %s left to fetch.' % (_fmt_gb(cached), _fmt_gb(need)))
    return dict(repo=repo, patterns=entry.get('fetch_patterns'), ignore=ignore, ggufs=ggufs, total=total, parts=parts, skipped=ignore,
                desc=' + '.join('%s (%s)' % (p[0], _fmt_gb(p[1])) for p in parts))


def _download_worker(plan, token, state):
    from huggingface_hub import snapshot_download, hf_hub_download
    try:
        paths, before = {}, {}
        for grepo, _gf in plan['ggufs']:
            d = _hub_dir(grepo)
            os.makedirs(d, exist_ok=True)
            before[grepo] = _du(d)
        bd = _hub_dir(plan['repo'])
        os.makedirs(bd, exist_ok=True)
        before[plan['repo']] = _du(bd)
        state['phase'] = 'snapshot'
        box, done = {}, threading.Event()
        def _run():
            try:
                kw = dict(repo_id=plan['repo'], token=token or None, max_workers=8)
                if plan['patterns']:
                    kw['allow_patterns'] = plan['patterns']
                if plan.get('ignore'):
                    kw['ignore_patterns'] = plan['ignore']
                box['snap'] = snapshot_download(**kw)
                for grepo, gfile in plan['ggufs']:
                    box[grepo] = hf_hub_download(repo_id=grepo, filename=gfile, token=token or None)
            except Exception as e:
                box['error'] = e
            finally:
                done.set()
        threading.Thread(target=_run, daemon=True).start()
        while not done.is_set():
            got = _du(bd) - before[plan['repo']]
            for grepo, _gf in plan['ggufs']:
                got += _du(_hub_dir(grepo)) - before[grepo]
            state['got'] = max(0, got)
            time.sleep(0.5)
        if 'error' in box:
            raise box['error']
        state['snap'] = box.get('snap')
        state['done'] = True
    except Exception as e:
        state['error'] = '%s: %s' % (type(e).__name__, str(e))
        ustatus('download failed — ' + state['error'])

# ── pipeline loader (signature-tolerant across WeeLLM classes) ─
PIPE_CACHE = {'key': None, 'pipe': None}
def _filtered_call(cls, model_dir, wanted):
    sig = inspect.signature(cls.from_pretrained)
    ps = sig.parameters
    variadic = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in ps.values())
    filt = {k: v for k, v in wanted.items() if (k in ps or variadic)}
    return cls.from_pretrained(model_dir, **filt)
def _load_worker(entry, quant, model_dir, dtype, vram_b, ram_b, state):
    try:
        import weellm
        cls = getattr(weellm, PIPE_CLS[entry['pipe']])
        kw = dict(device='cuda', torch_dtype=dtype, prefetch=(quant == 'full'), cache_to_ram=False, vae_tile_size=512, vram_budget=vram_b, ram_budget=ram_b)
        if quant != 'full':
            kw['transformer_path'] = '%s/%s' % (_rid(entry['gguf_repo_url']), quant)
            kw['prefetch'] = False
            if entry.get('te_gguf'):
                kw['text_encoder_path'] = '%s/%s' % (_rid(entry['te_gguf'][0]), entry['te_gguf'][1])
            if entry.get('te_safetensors'):
                kw['text_encoder_path'] = '%s/%s' % (_rid(entry['te_safetensors'][0]), entry['te_safetensors'][1])
        ustatus('loading %s (%s) …' % (entry['label'], quant))
        pipe = _filtered_call(cls, model_dir, kw)
        try:
            pipe.set_progress_bar_config(disable=True)
        except Exception:
            pass
        state['pipe'] = pipe
        state['done'] = True
        ustatus('pipeline ready: %s' % type(getattr(pipe, '_pipeline', pipe)).__name__)
    except Exception as e:
        state['error'] = '%s: %s' % (type(e).__name__, str(e))
        ustatus('load failed — ' + state['error'])
def do_unload():
    if PIPE_CACHE['pipe'] is not None:
        try:
            del PIPE_CACHE['pipe']
        except Exception:
            pass
        PIPE_CACHE.update(key=None, pipe=None)
    gc.collect()
    try:
        torch.cuda.synchronize()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    ustatus('model unloaded, VRAM released.')
    return status_html('Idle — model unloaded.', 'idle'), config_html(None, None, None)

# ── luxury HTML fragments (in-app) ─────────────────────────────
def status_html(msg, kind='run'):
    dot = {'run': '#e8c96a', 'ok': '#7ee2a0', 'err': '#ff9d9d', 'idle': '#9a94a3'}[kind]
    return ("<div style='display:flex;align-items:center;gap:10px;padding:10px 14px;border-radius:12px;" 
            "background:rgba(255,255,255,.03);border:1px solid rgba(212,175,55,.25);font-size:13px;color:#ece7d8;'>" 
            "<span class='luxdot' style='width:9px;height:9px;border-radius:50%;background:" + dot + ";box-shadow:0 0 10px " + dot + ";'></span><span>" + msg + "</span></div>")
def bar_html(pct, label):
    pct = max(0.0, min(100.0, pct))
    return ("<div style='font-size:12px;color:#9a94a3;margin:2px 0 6px;letter-spacing:.06em;'>" + label + "</div>" 
            "<div style='height:10px;border-radius:99px;background:rgba(255,255,255,.07);border:1px solid rgba(212,175,55,.25);overflow:hidden;'>" 
            "<div class='luxbar-anim' style='height:100%;width:" + ('%.1f' % pct) + "%;border-radius:99px;background:linear-gradient(90deg,#8a6d1c,#e8c96a);" 
            "transition:width .3s;'></div></div>")
def config_html(entry, quant, extra=None):
    if entry is None:
        return "<div style='font-size:12.5px;color:#8f8a9c;'>NO MODEL LOADED · CONFIGURE ABOVE AND PRESS RUN</div>"
    ex = extra or {}
    if quant == 'full':
        wlabel, wurl = 'Full repo', entry['repo_url']
    else:
        _furl = [u for v, l, u in entry['quants'] if v == quant]
        wlabel, wurl = 'Transformer ' + quant, (_furl[0] if _furl else entry['gguf_repo_url'])
    rows = [('Model', entry['label']),
            ('Weights', "<a style='color:#e8c96a;' href='" + wurl + "' target='_blank'>" + wlabel + '</a>'),
            ('Pipeline', PIPE_CLS[entry['pipe']]),
            ('Dtype', ex.get('dtype', '?')),
            ('VRAM / RAM budget', '%s / %s GB' % (ex.get('vram', '?'), ex.get('ram', '?'))),
            ('Prefetch', 'on' if quant == 'full' else 'off (GGUF)')]
    lis = ''.join("<div style='display:flex;justify-content:space-between;gap:14px;padding:5px 2px;border-top:1px solid rgba(255,255,255,.06);font-size:12.5px;'><span style='color:#9a94a3;'>%s</span><b style='color:#ece7d8;'>%s</b></div>" % r for r in rows)
    plan = ''.join("<div style='display:flex;justify-content:space-between;gap:14px;padding:5px 2px;border-top:1px solid rgba(255,255,255,.06);font-size:12.5px;'><span style='color:#9a94a3;'>fetch</span><b style='color:#ece7d8;'><a style='color:#e8c96a;' href='" + u + "' target='_blank'>" + l + '</a> · ' + _fmt_gb(b) + '</b></div>' for l, b, u in (ex.get('plan') or []))
    return "<div style='border:1px solid rgba(212,175,55,.25);border-radius:12px;padding:10px 14px;background:rgba(255,255,255,.02);'>" + lis + plan + '</div>'


# ── inference ──────────────────────────────────────────────────
def _native_supports(pipe, name):
    try:
        return name in inspect.signature(pipe._pipeline.__call__).parameters
    except Exception:
        return False
def _run_t2i(pipe, a, cb, use_cb):
    kw = dict(prompt=a['prompt'], height=a['h'], width=a['w'], num_inference_steps=a['steps'], seed=a['seed'])
    if a['guid'] > 0 and _native_supports(pipe, 'guidance_scale'):
        kw['guidance_scale'] = a['guid']
    if a['neg'] and _native_supports(pipe, 'negative_prompt'):
        kw['negative_prompt'] = a['neg']
    if use_cb and _native_supports(pipe, 'callback_on_step_end'):
        kw['callback_on_step_end'] = cb
    return pipe.generate(**kw)
def _run_edit(pipe, a, cb, use_cb):
    kw = dict(prompt=a['prompt'], image=a['init'], strength=a['strength'], height=a['h'], width=a['w'], num_inference_steps=a['steps'], seed=a['seed'])
    if a['guid'] > 0 and _native_supports(pipe, 'guidance_scale'):
        kw['guidance_scale'] = a['guid']
    if a['neg'] and _native_supports(pipe, 'negative_prompt'):
        kw['negative_prompt'] = a['neg']
    if use_cb and _native_supports(pipe, 'callback_on_step_end'):
        kw['callback_on_step_end'] = cb
    return pipe.generate(**kw)
def _run_video(pipe, entry, a, cb, use_cb):
    n = _snap73(a['frames']) if entry['pipe'] == 't2v' else _snap121(a['frames'])
    if n != a['frames']:
        ustatus('frames snapped %d → %d (architecture requirement)' % (a['frames'], n))
    kw = dict(prompt=a['prompt'], height=a['h'], width=a['w'], num_frames=n, num_inference_steps=a['steps'], seed=a['seed'], no_cache=True)
    if entry['pipe'] == 'i2v':
        kw['image'] = a['start']
    if a['guid'] > 0:
        kw['guidance_scale'] = a['guid']
    if use_cb:
        kw['callback_on_step_end'] = cb
    try:
        return pipe(**kw), n
    except TypeError as e:
        if 'callback_on_step_end' in str(e):
            ustatus('native pipeline drops step callbacks — progress will be coarse.')
            kw.pop('callback_on_step_end', None)
            return pipe(**kw), n
        raise
def _save_video_out(out, entry_label):
    from diffusers.utils import encode_video
    if hasattr(out, 'videos') and out.videos is not None:
        videos = out.videos[0] if isinstance(out.videos, list) else out.videos
    elif hasattr(out, 'video') and out.video is not None:
        videos = out.video[0] if isinstance(out.video, list) else out.video
    elif hasattr(out, 'frames') and out.frames is not None:
        videos = out.frames[0] if isinstance(out.frames, list) else out.frames
    else:
        raise RuntimeError('Unrecognized video output from %s' % entry_label)
    audio = getattr(out, 'audio', None)
    if isinstance(audio, list):
        audio = audio[0] if audio else None
    fps = getattr(out, 'fps', None) or 24
    path = os.path.join(OUT_DIR, 'out_%d.mp4' % int(time.time()))
    try:
        encode_video(videos, fps=fps, output_path=path, audio=audio, audio_sample_rate=getattr(out, 'sampling_rate', 24000))
    except Exception:
        import torchvision.io
        v = videos
        if isinstance(v, torch.Tensor):
            if v.dim() == 4 and v.shape[0] == 3:
                v = v.permute(1, 0, 2, 3)
            if v.dtype in (torch.float16, torch.bfloat16, torch.float32):
                v = (v * 255).clamp(0, 255).to(torch.uint8)
            torchvision.io.write_video(path, v.permute(0, 2, 3, 1), fps=24, audio_array=audio, audio_fps=24000, audio_codec='aac')
        else:
            raise
    return path

# ── master generator (streams status · bars · logs · media) ────
def _run_all_inner(task, model_key, quant, token, prompt, neg, w, h, steps, guid, seed, strength, frames, init_img, start_img, dtype_name, vram_b, ram_b):
    entry = MODELS.get(model_key)
    if entry is not None:  # Radio returns the label — map it back to the quant value
        _qmap = {lbl: val for val, lbl, _u in entry['quants']}
        quant = _qmap.get(quant, quant)
    if entry is None or not (prompt or '').strip():
        yield status_html('Pick a model and write a prompt first.', 'err'), config_html(None, None), bar_html(0, 'Download'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    if entry['pipe'] == 'edit' and init_img is None:
        yield status_html('Image Edit needs an input image.', 'err'), config_html(None, None), bar_html(0, 'Download'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    if entry['pipe'] == 'i2v' and start_img is None:
        yield status_html('Image → Video needs a start frame image.', 'err'), config_html(None, None), bar_html(0, 'Download'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    token = (token or '').strip()
    if entry['gated'] and not token and not entry.get('mirror_url'):
        ustatus('WARNING: %s is gated and has no mirror — a token will be required at download.' % entry['label'])
    seed = int(seed)
    if seed < 0:
        seed = random.randrange(2 ** 31 - 1)
    dtype = torch.float16 if dtype_name == 'float16' else torch.bfloat16
    a = dict(prompt=prompt.strip(), neg=(neg or '').strip(), w=int(w), h=int(h), steps=int(steps), guid=float(guid or 0), seed=seed, strength=float(strength), frames=int(frames), init=init_img, start=start_img)
    cfg = config_html(entry, quant, {'dtype': dtype_name, 'vram': vram_b, 'ram': ram_b, 'plan': None})
    if entry.get('supported') is False:
        _why = entry.get('unsupported_reason', 'This model cannot be loaded by WeeLLM yet.')
        ustatus('refused — ' + _why)
        yield status_html('Cannot run %s: %s' % (entry['label'], _why), 'err'), cfg, bar_html(0, 'Download'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    ustatus('run: %s · %s · %dx%d · %d steps · seed %d' % (entry['label'], quant, a['w'], a['h'], a['steps'], seed))

    def snap():
        return status_html('WORKING — SEE BARS + LOGS BELOW', 'run'), cfg, bar_html(dl_state.get('pct', 0), dl_state.get('txt', 'Download')), bar_html(gen_state.get('pct', 0), gen_state.get('txt', 'Generation')), _log_text(), None, None

    # — phase 1: download (HF Hub only) —
    dl_state = {'pct': 0, 'txt': 'Download — resolving sizes…'}
    gen_state = {'pct': 0, 'txt': 'Generation — waiting'}
    yield snap()
    try:
        plan = plan_download(entry, quant, token)
    except Exception as e:
        _msg = str(e)
        if '401' in _msg or 'Unauthorized' in _msg:
            _msg += ' — this repo is gated: paste an HF token above AND click Agree/Accept on the repo page (%s).' % (_rid(entry['repo_url']))
        ustatus('preflight failed — %s' % _msg)
        yield status_html('Download blocked: %s' % _msg, 'err'), cfg, bar_html(0, 'Download'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    if not plan['total']:
        ustatus('WARNING: the Hub did not report file sizes — downloading without a progress denominator.')
    for _plabel, _pbytes, _purl in plan['parts']:
        ustatus('fetch plan: %s = %s <%s>' % (_plabel, _fmt_gb(_pbytes), _purl))
    if plan.get('skipped'):
        ustatus('skipped as assumed-unused: ' + ', '.join(plan['skipped']))
    dl = {'got': 0, 'done': False, 'error': None, 'snap': None}
    threading.Thread(target=_download_worker, args=(plan, token, dl), daemon=True).start()
    _warned_over = False
    while not dl['done'] and not dl.get('error'):
        denom = max(plan['total'], dl['got'])  # Hub listing can lag real bytes; never show >100%
        pct = 100.0 * min(1.0, dl['got'] / max(1, denom))
        dl_state.update(pct=pct, txt='Download — %s / %s' % (_fmt_gb(dl['got']), _fmt_gb(denom)))
        if not _warned_over and dl['got'] > plan['total'] and plan['total'] > 0:
            _warned_over = True
            ustatus('note: fetched %s vs %s planned — Hub sizes lagged; extra bytes are harmless cache, continuing.' % (_fmt_gb(dl['got']), _fmt_gb(plan['total'])))
        yield snap()
        time.sleep(0.4)
    if dl.get('error'):
        yield status_html('Download failed: %s' % dl['error'], 'err'), cfg, bar_html(dl_state['pct'], 'Download — failed'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    dl_state.update(pct=100, txt='Download — complete (%s)' % _fmt_gb(plan['total']))
    model_dir = dl['snap']
    if entry.get('subfolder'):
        model_dir = os.path.join(model_dir, entry['subfolder'])
    ustatus('weights at %s' % model_dir)
    cfg = config_html(entry, quant, {'dtype': dtype_name, 'vram': vram_b, 'ram': ram_b, 'plan': plan['parts']})
    yield snap()

    # — phase 2: load (cached, instant on repeat runs) —
    key = (model_key, quant, dtype_name, vram_b, ram_b)
    if PIPE_CACHE['key'] != key or PIPE_CACHE['pipe'] is None:
        if PIPE_CACHE['pipe'] is not None:
            ustatus('pipeline config changed — rebuilding automatically, no manual unload needed.')
            do_unload()
        gen_state.update(pct=0, txt='Generation — loading pipeline…')
        yield snap()
        ls, _tries = {}, 0
        while True:
            _tries += 1
            ls = {}
            threading.Thread(target=_load_worker, args=(entry, quant, model_dir, dtype, vram_b, ram_b, ls), daemon=True).start()
            while not ls.get('done') and not ls.get('error'):
                yield snap()
                time.sleep(0.4)
            if ls.get('error') and 'out of memory' in ls['error'].lower() and _tries == 1:
                ustatus('load hit CUDA OOM — purging VRAM and retrying once…')
                do_unload()
                continue
            break
        if ls.get('error'):
            yield status_html('Load failed: %s' % ls['error'], 'err'), cfg, bar_html(100, 'Download — complete'), bar_html(0, 'Generation'), _log_text(), None, None
            return
        PIPE_CACHE.update(key=key, pipe=ls['pipe'])
    else:
        ustatus('reusing cached pipeline (sampling-only change — no rebuild needed).')
    pipe = PIPE_CACHE['pipe']

    # — phase 3: generate —
    gstate, gbox = {'step': 0, 'total': a['steps']}, {}
    def _cb(_p, i, _t, cb_kwargs):
        try:
            gstate['step'] = int(i) + 1
        except Exception:
            pass
        return cb_kwargs
    gen_state.update(pct=1, txt='Generation — denoising 0/%d' % a['steps'])
    yield snap()
    t0 = time.time()
    try:
        torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass
    def _infer():
        try:
            if entry['pipe'] == 't2i':
                gbox['img'] = _run_t2i(pipe, a, _cb, True)
            elif entry['pipe'] == 'edit':
                gbox['img'] = _run_edit(pipe, a, _cb, True)
            else:
                out, _n = _run_video(pipe, entry, a, _cb, True)
                gbox['vid'] = _save_video_out(out, entry['label'])
        except Exception as e:
            gbox['error'] = '%s: %s' % (type(e).__name__, str(e))
            ustatus('inference failed — ' + gbox['error'])
            ustatus(traceback.format_exc(limit=5)[-1500:])
    threading.Thread(target=_infer, daemon=True).start()
    while 'img' not in gbox and 'vid' not in gbox and 'error' not in gbox:
        pct = 100.0 * min(1.0, gstate['step'] / max(1, gstate['total']))
        gen_state.update(pct=pct, txt='Generation — denoising %d/%d' % (min(gstate['step'], gstate['total']), gstate['total']))
        yield snap()
        time.sleep(0.4)
    if 'error' in gbox:
        if 'out of memory' in gbox['error'].lower():
            do_unload()
            ustatus('VRAM exhausted — model unloaded automatically. Lower resolution/steps or raise the VRAM budget, then Run again.')
        yield status_html('Generation failed: %s' % gbox['error'], 'err'), cfg, bar_html(100, 'Download — complete'), bar_html(gen_state['pct'], 'Generation — failed'), _log_text(), None, None
        return
    dt = time.time() - t0
    try:
        peak = torch.cuda.max_memory_allocated() / 1e9
    except Exception:
        peak = -1.0
    img, vid = gbox.get('img'), gbox.get('vid')
    _blank = False
    if img is not None:
        img.save(os.path.join(OUT_DIR, 'img_%d.png' % int(t0)))
        try:
            _ex = img.convert('RGB').getextrema()
            _blank = all((mx - mn) == 0 for mn, mx in _ex)
            if _blank:
                ustatus('WARNING: output is one flat color (classic NaN black-image failure) — re-run with dtype bfloat16.')
        except Exception:
            pass
    ustatus('done in %.0fs · peak %.2f GB · seed %d' % (dt, peak, seed))
    gen_state.update(pct=100, txt='Generation — done in %.0fs' % dt)
    _done_txt = 'Done in %.0fs · peak %.2f GB VRAM · seed %d' % (dt, peak, seed)
    _done_kind = 'ok'
    if _blank:
        _done_txt += ' — WARNING: output looks blank. Re-run with dtype bfloat16.'
        _done_kind = 'err'
    yield status_html(_done_txt, _done_kind), cfg, bar_html(100, 'Download — complete'), bar_html(100, gen_state['txt']), _log_text(), img, vid

_RUN_LOCK = threading.Lock()

def run_all(task, model_key, quant, token, prompt, neg, w, h, steps, guid, seed, strength, frames, init_img, start_img, dtype_name, vram_b, ram_b):
    # One run at a time: a second RUN click while downloading/loading would
    # build a whole second pipeline next to the first and OOM the box.
    if not _RUN_LOCK.acquire(blocking=False):
        yield status_html('Another run is already in progress — please wait for it to finish.', 'err'), config_html(None, None), bar_html(0, 'Download'), bar_html(0, 'Generation'), _log_text(), None, None
        return
    try:
        yield from _run_all_inner(task, model_key, quant, token, prompt, neg, w, h, steps, guid, seed, strength, frames, init_img, start_img, dtype_name, vram_b, ram_b)
    finally:
        _RUN_LOCK.release()


# ── UI events ────────────────────────────────────────────────
def _models_for(task):
    return [(m['label'], k) for k, m in MODELS.items() if m['task'] == task]
def on_task(task):
    opts = _models_for(task)
    first = opts[0][1] if opts else None
    e = MODELS[first]
    import gradio as gr
    qdefault = [l for v, l, _u in e['quants'] if v == e['default_q']][0]
    return (gr.update(choices=opts, value=first), gr.update(choices=[l for _v, l, _u in e['quants']], value=qdefault), gr.update(visible=(task == 'edit')), gr.update(visible=(task in ('t2v', 'i2v'))), gr.update(visible=(task == 'i2v')), gr.update(value=e['steps']), gr.update(value=e['w']), gr.update(value=e['h']), gr.update(value=e.get('frames', 73)), model_hint(e))
def on_model(model_key):
    import gradio as gr
    e = MODELS[model_key]
    return (gr.update(choices=[l for _v, l, _u in e['quants']], value=[l for v, l, _u in e['quants'] if v == e['default_q']][0]), gr.update(value=e['steps']), gr.update(value=e['w']), gr.update(value=e['h']), gr.update(value=e.get('frames', 73)), model_hint(e))
def model_hint(e):
    return ("<div style='font-size:12px;color:#9a94a3;'>%s — <a class='mono' style='color:#e8c96a;' href='%s' target='_blank'>%s</a><br>%s</div>" % (e['label'], e['repo_url'], _rid(e['repo_url']), e['access']))


# ---- version-tolerant component constructor ----
# Drops cosmetic kwargs the installed Gradio does not know (e.g.
# show_copy_button), so the UI builds on old AND new Gradio alike.
def _C(cls, *args, **kw):
    try:
        _ps = inspect.signature(cls).parameters
        if not any(p.kind == inspect.Parameter.VAR_KEYWORD for p in _ps.values()):
            kw = {k: v for k, v in kw.items() if k in _ps}
    except Exception:
        pass
    try:
        return cls(*args, **kw)
    except TypeError:
        for _drop in ('show_copy_button', 'autoscroll', 'max_lines', 'placeholder'):
            kw.pop(_drop, None)
        return cls(*args, **kw)

# ── build the app ────────────────────────────────────────────
import gradio as gr
_auto_vram, _auto_ram = _budgets()
GRADIO_CSS = ('.gradio-container{max-width:1600px !important;width:100% !important;margin:0 auto !important;padding-left:20px !important;padding-right:20px !important;background:radial-gradient(1200px 500px at 50% -8%,rgba(212,175,55,.10),transparent),linear-gradient(180deg,#0c0c13,#101018) !important;color:#ece7d8 !important;}'
 '.gradio-container h1{font-family:Georgia,serif !important;font-weight:400 !important;letter-spacing:.28em !important;font-size:26px !important;color:#e8c96a !important;text-align:center !important;margin-bottom:0 !important;}'
 '.subtitle{text-align:center;color:#9a94a3;font-size:12.5px;letter-spacing:.12em;margin:2px 0 14px;}'
 '.gr-button-primary{background:linear-gradient(135deg,#f3d97b,#c9a227) !important;color:#0b0b12 !important;font-weight:800 !important;letter-spacing:.08em !important;border:none !important;border-radius:12px !important;box-shadow:0 6px 22px rgba(212,175,55,.35) !important;}'
 '.gr-button-secondary{border:1px solid rgba(212,175,55,.4) !important;color:#e8c96a !important;border-radius:12px !important;background:rgba(212,175,55,.06) !important;}'
 '.gr-panel,.gr-box,.gr-form{background:rgba(255,255,255,.025) !important;border:1px solid rgba(255,255,255,.08) !important;border-radius:14px !important;}'
 '.gr-input,.gr-text-input,textarea{background:rgba(0,0,0,.35) !important;color:#ece7d8 !important;}'
 'footer{display:none !important;}'
 ' .gradio-container label span,.gradio-container legend span,.gradio-container .gr-button{text-transform:uppercase !important;letter-spacing:.12em !important;}'
 ' .luxbar-anim{position:relative;overflow:hidden;}'
 ' .luxbar-anim::after{content:"";position:absolute;inset:0;background:linear-gradient(100deg,transparent 20%,rgba(255,255,255,.35) 50%,transparent 80%);animation:luxsheen 1.8s linear infinite;}'
 ' @keyframes luxsheen{from{transform:translateX(-100%)}to{transform:translateX(100%)}}'
 ' .luxdot{animation:luxpulse 1.6s ease-in-out infinite;}'
 ' @keyframes luxpulse{0%,100%{opacity:1}50%{opacity:.35}}'
 '.mono{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;color:#cfc9b8;}')
with gr.Blocks(title='WeeLLM Studio', analytics_enabled=False, css=GRADIO_CSS, theme=gr.themes.Soft(primary_hue='amber', neutral_hue='stone')) as demo:
    _C(gr.HTML, "<h1>WEELLM STUDIO</h1><div class='subtitle'>LAYER-STREAMED DIFFUSION · LOW-VRAM CINEMA</div>")
    with _C(gr.Row):
        task_r = _C(gr.Radio, choices=TASKS, value='t2i', label='1 · INFERENCE TYPE')
        model_d = _C(gr.Dropdown, choices=_models_for('t2i'), value='z-image-turbo', label='2 · MODEL')
    with _C(gr.Row):
        quant_r = _C(gr.Radio, choices=[l for _v, l, _u in MODELS['z-image-turbo']['quants']], value=[l for v, l, _u in MODELS['z-image-turbo']['quants'] if v == MODELS['z-image-turbo']['default_q']][0], label='3 · QUANTIZATION — TRANSFORMER ONLY · ENCODER STAYS FULL PRECISION')
        token_t = _C(gr.Textbox, label='HF TOKEN (GATED REPOS ONLY)', type='password', placeholder='hf_…', scale=1)
    hint_h = _C(gr.HTML, model_hint(MODELS['z-image-turbo']))
    with _C(gr.Accordion, '4 · PROMPT & SAMPLING', open=True):
        prompt_t = _C(gr.Textbox, label='PROMPT', lines=3, placeholder='A serene Japanese zen garden at sunrise, photorealistic…')
        neg_t = _C(gr.Textbox, label='NEGATIVE PROMPT (IGNORED BY DISTILLED TURBOS)', lines=1, placeholder='blurry, low quality')
        with _C(gr.Row):
            w_s = _C(gr.Slider, 256, 2048, value=1024, step=16, label='WIDTH')
            h_s = _C(gr.Slider, 256, 2048, value=1024, step=16, label='HEIGHT')
            steps_s = _C(gr.Slider, 1, 50, value=8, step=1, label='STEPS')
        with _C(gr.Row):
            guid_s = _C(gr.Slider, 0, 10, value=1.0, step=0.5, label='GUIDANCE (0 = PIPELINE DEFAULT)')
            seed_n = _C(gr.Number, value=42, precision=0, label='SEED (−1 = RANDOM)')
            dtype_r = _C(gr.Radio, choices=['bfloat16', 'float16'], value='bfloat16', label='DTYPE (BFLOAT16 = SAFE · FLOAT16 = FASTER BUT CAN NAN TO BLACK)')
        with _C(gr.Row):
            vram_s = _C(gr.Slider, 2, 16, value=_auto_vram, step=0.5, label='VRAM BUDGET (GB)')
            ram_s = _C(gr.Slider, 2, 24, value=_auto_ram, step=0.5, label='RAM BUDGET (GB)')
        with _C(gr.Row, visible=False) as edit_box:
            init_im = _C(gr.Image, type='pil', label='INPUT IMAGE (EDIT)')
            str_s = _C(gr.Slider, 0.1, 1.0, value=0.8, step=0.05, label='DENOISE STRENGTH')
        with _C(gr.Row, visible=False) as vid_box:
            frames_s = _C(gr.Slider, 9, 241, value=73, step=1, label='FRAMES (AUTO-SNAPPED · H3 = 17K+5 · LTX = 8K+1)')
        with _C(gr.Row, visible=False) as i2v_box:
            start_im = _C(gr.Image, type='pil', label='START FRAME (LTX IMAGE → VIDEO)')
    with _C(gr.Row):
        run_b = _C(gr.Button, '✦  RUN  ✦', variant='primary', scale=3)
        unload_b = _C(gr.Button, 'UNLOAD MODEL', variant='secondary', scale=1)
    cfg_h = _C(gr.HTML, config_html(None, None))
    status_h = _C(gr.HTML, status_html('IDLE — CONFIGURE AND PRESS RUN', 'idle'))
    with _C(gr.Row):
        dl_h = _C(gr.HTML, bar_html(0, 'Download'))
        gen_h = _C(gr.HTML, bar_html(0, 'Generation'))
    logs_t = _C(gr.Textbox, label='LIVE LOGS', lines=18, max_lines=30, autoscroll=True, show_copy_button=True, value='— PRESS RUN · LOGS STREAM HERE —')
    with _C(gr.Row):
        img_o = _C(gr.Image, label='IMAGE RESULT', interactive=False)
        vid_o = _C(gr.Video, label='VIDEO RESULT')
    _C(gr.HTML, "<div class='subtitle'>OUTPUTS SAVE TO /content/weellm_studio · KEEP THIS RUNTIME ALIVE WHILE GENERATING</div>")
    task_r.change(on_task, inputs=[task_r], outputs=[model_d, quant_r, edit_box, vid_box, i2v_box, steps_s, w_s, h_s, frames_s, hint_h])
    model_d.change(on_model, inputs=[model_d], outputs=[quant_r, steps_s, w_s, h_s, frames_s, hint_h])
    unload_b.click(do_unload, inputs=[], outputs=[status_h, cfg_h])
    with _C(gr.Accordion, 'HOW MODEL + QUANT DOWNLOADS WORK', open=False):
        _C(gr.Markdown, 'A **quant** (Q4_K_M / Q6_K / Q8_0) replaces **only the diffusion transformer** — the weights streamed every denoising step. The **text encoder, VAE, scheduler and configs always come full-precision from the base repo**, whose transformer shards are skipped and never downloaded. Example: Z-Image-Turbo Q4_K_M fetches about 8.5 GB of base parts plus the 4.5 GB transformer instead of 32.9 GB. Consequence: when the **base repo is gated** (FLUX.2-Klein, Ideogram, LTX), a token is still required in quant mode — the encoder and VAE live behind the gate. Krea is the exception: its open mirror covers the base parts when no token is given. LTX 2.5 INT8-ConvRot (official) and W4A8-ConvRot (Winnougan) are first-class: WeeLLM decodes them block-by-block in pure torch (same numbers as ComfyUI, no extra install). A **quant** replaces **only the diffusion transformer**; for the two ConvRot entries the matching quantized text encoder is fetched automatically, while **VAEs/scheduler/configs always come full-precision from the base repo**. Example: W4A8 fetches the 12.5 GB DiT + 10.6 GB Gemma4 encoder + ~17 GB of base parts (~40 GB total) instead of ~127 GB. Free-Colab rule: any option totalling over 60 GB is removed from the menus (LTX-2.5 full, LTX-2 Q8, LTX-2 entry, MiniMax full). The downloader also skips weights that are never executed: replaced text encoders (configs still fetch) and the LTX prompt enhancer.') 
    _click_kw = dict(inputs=[task_r, model_d, quant_r, token_t, prompt_t, neg_t, w_s, h_s, steps_s, guid_s, seed_n, str_s, frames_s, init_im, start_im, dtype_r, vram_s, ram_s], outputs=[status_h, cfg_h, dl_h, gen_h, logs_t, img_o, vid_o])
    try:
        run_b.click(run_all, concurrency_limit=1, **_click_kw)
    except TypeError:
        run_b.click(run_all, **_click_kw)

# ── launch: capture every stray print, show ONLY the link card ─
_boxh = shell_box('WeeLLM Studio', "<span class='pill run'>STARTING</span>", 'Cell 2 of 2 — building the app and opening a share tunnel…')
_cap_out, _cap_err = io.StringIO(), io.StringIO()
try:
    with contextlib.redirect_stdout(_cap_out), contextlib.redirect_stderr(_cap_err):
        try:
            demo.queue(max_size=8)
        except TypeError:
            demo.queue()
        demo.launch(server_name='0.0.0.0', share=True, inline=False, quiet=True, prevent_thread_lock=True, show_error=False)
    ustatus('app launched.')
    _url = getattr(demo, 'share_url', None)
    shell_box('WeeLLM Studio', "<span class='pill ok'>LIVE</span>", 'The app is running — open the public link below. Keep this tab and runtime alive; the heartbeat below holds the session.', link=_url, handle=_boxh)
    _t0, _beats = time.time(), 0
    while True:  # keep-alive: hold the cell (and the Colab session) awake while serving
        time.sleep(60)
        _beats += 1
        _up = int(time.time() - _t0)
        try:
            _vram = torch.cuda.memory_allocated() / 1e9
        except Exception:
            _vram = -1.0
        shell_box('WeeLLM Studio', "<span class='pill ok'>LIVE</span>", 'Heartbeat #%d · uptime %dh %02dm · VRAM in use %.2f GB · serving — stop the cell to shut down.' % (_beats, _up // 3600, (_up % 3600) // 60, _vram), link=_url, handle=_boxh)
except KeyboardInterrupt:
    shell_box('WeeLLM Studio', "<span class='pill bad'>STOPPED</span>", 'Keep-alive interrupted — Studio stopped. Re-run this cell to restart.', handle=_boxh)
except Exception as e:
    shell_box('WeeLLM Studio', "<span class='pill bad'>ERROR</span>", 'Launch failed: %s: %s' % (type(e).__name__, e), handle=_boxh)
